# VEP

Variant Effect Prediction (VEP) using Protein Language Models.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True

import src.utils as utils
import src.config as config
import src.haplosaurus as hs
import src.ESM as ESM
import src.ESM_predict as ESMp
import src.gprofiler as gp
import src.vep_pipeline as vp
import src.vep_analysis as va
import src.vep_metrics as vm
import src.proteingym as pg 
import src.ensembl_rest as er
import src.biopython as bp

pd.set_option('display.max_columns', None)

## Import data

### Candidate proteins
A curated list of proteins that clinically relevant and incompletely penetrant across different ethnic populations.


### ProteinGym clinical mutations
Import set of clinical variants (substitutions and indels) to use for perturbing the wild-type protein sequences *in silico*.

In [ ]:
proteins_df = pg.merge_resources(rows_per_id=None)
proteins_df

In [ ]:
candidate_proteins = utils.get_candidate_proteins()
candidate_proteins['protein'] = candidate_proteins['RefSeq Protein']
# candidate_proteins = er.map_ids(df=candidate_proteins,
#                                 input_col="RefSeq Protein")
candidate_proteins = pg.map_resources(df=candidate_proteins)
candidate_proteins

### Haplotypes
Import haplotype sequences from Haplosaurus. 
These are the wild-type sequences for each protein that occurs within the 1000 Genomes Project.






In [ ]:
haplotypes_hgdp = hs.get_haplotypes(cache = hs.DIR_DICT["HGDP_haplotypes"],
                                     tx_ids=proteins_df['ENST'].unique(), 
                                    cache_only = True) 
haplotypes_1kg = hs.get_haplotypes(cache = hs.DIR_DICT["haplotypes"],
                                    tx_ids=proteins_df['ENST'].unique(), 
                                    cache_only = True) 
haplotypes = hs.merge_haplotype_datasets({"1KG":haplotypes_1kg, "HGDP":haplotypes_hgdp},
                                         tx_id_method = "union",
                                         use_deepcopy = True) 

Plot the number of haplotypes per protein, as well as the number of WT variants per haplotype.

In [ ]:
hs.plot_haplotypes_summary(haplotypes)

Map haplotypes back to samples (individuals).

In [ ]:
haps_to_samples = hs.haplotypes_to_samples(haplotypes=haplotypes,  
                                           as_df=True,
                                           add_sample_metadata=True,
                                           return_seqs=False, 
                                           add_ref=False, 
                                           duplicate_ref=False,
                                           verbose=True,
                                           )
haps_to_samples.head()

Infer per-superpopulation haplotype frequencies from the haplotype-sample maps.
--- 


For each haplotype, `calculate_haplotype_frequency_per_superpop()` assigns a:
- `***_freq`: The frequency at which the haplotype occurs in each superopulation.
- `***_samples`: The number of samples in which the haplotype occurs in each superopulation.
- `top_superpopulation`: The superpopulation in which the haplotype occurred most frequently.
- `top_superpopulation_freq`: The frequency at which the happlotype occurs in the `top_superpopulation`.
- `superpopulation_count`: The number of unique superpopulations in which the haplotype occurs.
- `specific_superpopulation`: If the haplotype only occurred in one superopulation, this column is filled with its name. 
- `specific_superpopulation_samples`: If the haplotype only occurred in one superopulation, this column is filled with the number of samples from that superpopulation. 

*NOTE*: In this case, each individual has two "samples" (one per haplotype).

In [ ]:
freq_df.to_parquet("results/data/freq_df.parquet")

In [ ]:
freq_df = hs.calculate_haplotype_frequency_per_superpop(
    haps_to_samples,
    cast=True, 
    verbose=True,
    add_specificity_cols=True
)
# freq_df.to_parquet("results/data/freq_df.parquet")
freq_df.head()

### Create Upset plot

In [ ]:
upset, counts_df = hs.plot_superpop_upset(freq_df)

In [ ]:
if 'specific_superpopulation' not in locals():
    specific_superpopulation = freq_df['specific_superpopulation'].copy()
freq_df.loc[freq_df['specific_superpopulation_samples']< 2, "specific_superpopulation"] = pd.NA 

Plot the number of samples per superpopulation, as well as whether each haplotype is specific to one superpopulation.

In [ ]:
plot_haplotypes_and_superpop_bar_out = hs.plot_haplotypes_and_superpop_bar(freq_df,
                                                                  haplotypes,
                                                                 width_ratios=(.2,1), figsize=(13,5))

`filter_prot_df` provides a convenient way to filter the ProteinGym mutations according to which ones are present in the 1KG haplotypes data,   
as well as which ones are present in the candidate proteins.

It will automatically figure out which `proteins_df` columns in it should be filtering on according to the prefixes in the `protein_ids` ('ENSP' or 'ENST' or 'protein') and `haplotypes` ('ENSP' or 'ENST').

It can also filter by the types of mutations you wish to inject during the VEP procedure (see the 'source_type' column for available options).

The final product is the filtered `proteins_df`, which we store as a new DataFrame `prot_df`.

In [ ]:
prot_df = vp.filter_prot_df(proteins_df,
#                             protein_ids=candidate_proteins["ENSP"],
                            haplotypes=haplotypes,
                            source_types="clinical_ProteinGym_substitutions",
                            )
prot_df

We also need to remove any suspicious proteins that have a different reference sequence lengths in the Haplosaurus haplotypes and the ProteinGym mutations   
(which suggests that the ProteinGym proteins are not exactly the same as the Haplosaurus ones).

In [ ]:
prot_df = vp.add_sequence_checks(prot_df,
                                  haplotypes=haplotypes)
prot_df = prot_df.loc[prot_df['proteingym_haplosaurus_seq_identical'] == True]
prot_df

## Run VEP pipeline

Run the VEP pipeline which iterates VEP over a series of models/proteins/haplotypes/variant source types/scoring strategies. 

The results are saved in a folder structure:  
{save_dir}/  
    {model}/  
    {protein_id}/  
        {haplotype}/  
        {source_type}/  
            {scoring_strategy1}.csv.gz  
            {scoring_strategy2}.csv.gz  
            {scoring_strategy3}.csv.gz  

List all models currently supported by the VEP pipeline.

In [ ]:
vp.list_models(as_list=False)

Run VEP pipeline and store the results in `save_dir`.

In [ ]:
import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True

import src.vep_pipeline as vp 
import src.haplosaurus as hs
import torch
torch.cuda.set_device("cuda:0")

save_paths = vp.vep_pipeline(
    # hap_dir = hs.DIR_DICT["HGDP_haplotypes"],
    # prot_df = prot_df,
    # haplotypes = haplotypes, 
    models = [
        # "esm1_t6_43M_UR50S",
          # "esm1_t34_670M_UR50D",
          # "esm1v_t33_650M_UR90S_1",
          # "esm1b_t33_650M_UR50S",
        #   "esm2_t12_35M_UR50D",
          # "esm2_t33_650M_UR50D",
        #   "esm2_t36_3B_UR50D"
          "esm2_t48_15B_UR50D",
      # "esmfold_v1",
            # "esm3_sm_open_v1",
            # "esmc_600m"
             ],
    scoring_strategies = [
        # "wt-marginals", 
        # "masked-marginals",
        "masked-marginals",
        # "pseudo-ppl" # Takes much longer to run
        ],
    enable_data_parallel=False,
    source_types = ["clinical_ProteinGym_substitutions"],
    verbose = False
)

print(len(save_paths),"results files generated.")

## Explore VEP results

Automatically search for, import, and merge the VEP results. 
You can import only subsets of results by setting the save_dir to a lower level, e.g.: 
`os.path.join(save_dir,{model_name},{protein_id}})`

Setting `add_metadata=True` will automatically import additional variant-level metadata (collected from ProteinGym) which will be helpful for downstream plotting/analysis tasks.

### Merge VEP results files
Merge individual VEP files into one large dataframe.

In [ ]:
vep_files = va.list_vep_files(as_df=True)

Gather all VEP results for each model and save as one parquet each.

In [ ]:
# for model_location in vep_files["model_location"].unique():
#     print(model_location)
#     save_path = f"results/data/vep_df_{model_location}.parquet"
#     if os.path.exists(save_path):
#         print(f"Skipping {model_location} because it already exists")
#         continue
#     vep_df = va.merge_vep(add_metadata=False, 
#                           vep_files=vep_files[vep_files["model_location"] == model_location],
#                           scoring_strategy=["masked-marginals"])
#     vep_df.to_parquet(save_path)

...Or download a subset of the data for just one model.

### Check whether the VEP is affected by whether the variant is already in the haplotype

In [ ]:
import seaborn as sns

vep_df = utils.sort_by_clinsig(vep_df)
g = sns.FacetGrid(data=vep_df, 
                   row='clinsig',    
                   sharey=False,
                   height=4, aspect=1.2)
g.map_dataframe(sns.violinplot, 
                x='mutant_in_haplotype', 
                y='VEP',
                hue='clinsig',
                palette=utils.get_clinsig_palette())
g.tight_layout()

### Check out of frame mutants

These should all be filtered out automatically by the vep_pipeline, but just to check.

In [ ]:
# Import haplotypes if not already defined
if 'haplotypes' not in locals():
    haplotypes = hs.get_haplotypes(
        tx_ids=vep_df['ENST'].unique().tolist(),
        cache_only=True
    )

# Add haplotype sequences to the VEP dataframe
vep_df = va.add_mutant_out_of_frame(vep_df, 
                                    haplotypes=haplotypes,
                                    force=True)


# Check if any haplotypes with variants that are out of frame
print(vep_df.loc[vep_df['mutant_out_of_frame']==True]['haplotype'].unique().tolist()[:10])


# Filter out variants that are out of frame, as they should not not be able to produce a VEP
# vep_df = vep_df.loc[vep_df['mutant_out_of_frame']==False]
vep_df.head()

### Count haplotypes where the variant is already in the WT sequence

In [ ]:
vep_df.loc[vep_df['mutant_in_haplotype']==True].groupby(['clinsig'])['haplotype'].nunique()

In [ ]:
print(vep_df['haplotype_sequence_len_pct'].hist(bins=50))
print(vep_df['haplotype_sequence_len_pct'].describe())


In [ ]:
# Plot haplotype_sequence_len_pct vs VEP as point density with regression line
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats


# Get the per-protein mean to reduce the size of the dataset
# Downsample to 1000 random rows per protein
# plot_data = vep_df.loc[vep_df['mutant_out_of_frame']==False].sample(100000, random_state=42)
plot_data = vep_df.groupby(['model_location','protein','haplotype','haplotype_sequence_len_pct','clinsig'], observed=True).agg({'VEP': 'mean'}).reset_index()
max_pct = .95
plot_data = plot_data.loc[plot_data['haplotype_sequence_len_pct'] < max_pct]
print(plot_data.shape)
plot_data.head()

# Create a categorical column with the defined order
plot_data['clinsig'] = pd.Categorical(
    plot_data['clinsig'], 
    categories=list(utils.get_clinsig_palette().keys())[::-1], 
    ordered=True
)
# Sort the dataframe by the ordered categorical column
plot_data = plot_data.sort_values('clinsig')

min_vep = plot_data['VEP'].min()
 
# Calculate histogram data for the distribution subplot
hist_data = plot_data.copy()
# Create bins at 5% intervals
bins = np.arange(0, max_pct + 0.05, 0.05)
hist_data['bin'] = pd.cut(hist_data['haplotype_sequence_len_pct'], bins=bins)
# Count unique haplotypes in each bin
haplotype_counts = hist_data.groupby('bin')['haplotype'].nunique().reset_index()
haplotype_counts['bin_center'] = haplotype_counts['bin'].apply(lambda x: x.mid)

# Create new plot with the correct number of rows for the grid
# Get the number of categories to set the correct height_ratios
num_categories = len(plot_data['clinsig'].cat.categories)
g = sns.FacetGrid(plot_data, 
                row='clinsig',  
                sharex=True,
                sharey=True, 
                aspect=1.2,
                height=3)  # Remove the incorrect height_ratios parameter

# Map the custom function to the FacetGrid
g.map_dataframe(sns.kdeplot,
                 x="haplotype_sequence_len_pct",
                 y="VEP", 
                 fill=True, 
                 thresh=0, levels=15, cmap="mako",
                 bw_adjust=1, 
                 clip=[(0, max_pct), (min_vep, None)],  # Ensure x-axis doesn't go below 0
                 extend="both",
                 alpha=1.0
                 )
g.map_dataframe(sns.regplot,
                x="haplotype_sequence_len_pct",
                y="VEP", 
                scatter=False,
                lowess=True,
                n_boot=1000,
                # robust=True,
                ci=95,  # Add confidence interval
                line_kws={"color": "white", "linewidth": 2},
                )

# Calculate and add Pearson correlation for each subplot
for i, clinsig in enumerate(plot_data['clinsig'].cat.categories):
    subset = plot_data[plot_data['clinsig'] == clinsig].dropna(subset=['haplotype_sequence_len_pct', 'VEP'])
    # r, p = stats.spearmanr(subset['haplotype_sequence_len_pct'], subset['VEP'])
    # r2 = r**2 

    import numpy as np
    import statsmodels.api as sm
    from sklearn.metrics import mean_squared_error


    # Sample data (replace with your actual data)
    x_actual = subset['haplotype_sequence_len_pct']
    y_actual = subset['VEP']

    # Fit LOESS model
    lowess = sm.nonparametric.lowess(y_actual, x_actual, frac=0.3)
    x_predicted = lowess[:, 0]
    y_predicted = lowess[:, 1]

    # Calculate SS_res
    residuals = y_actual - y_predicted
    ss_res = np.sum(residuals**2)

    # Calculate SS_tot
    y_mean = np.mean(y_actual)
    ss_tot = np.sum((y_actual - y_mean)**2)

    # Calculate pseudo R-squared
    r2 = 1 - (ss_res / ss_tot)

    # Calculate MSE
    mse = mean_squared_error(y_actual, y_predicted)
    
    ax = g.axes[i, 0]
    ax.text(0.05, 0.95, f'coef = {r2:.3f}, MSE = {mse:.3f}', 
            transform=ax.transAxes, 
            fontsize=10, 
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

# Set x-axis limits for all subplots to ensure they don't go below 0
g.set(xlim=(0, max_pct))
# Create a separate figure for the histogram with the same width as the facet grid
fig_width, _ = g.fig.get_size_inches()
plt.figure(figsize=(fig_width, 1.5))

# Create the bar plot
bars = plt.bar(haplotype_counts['bin_center'], haplotype_counts['haplotype'], 
        width=0.04, color='darkblue', alpha=0.7)

# Add a line connecting the tops of the bars
bar_heights = haplotype_counts['haplotype']
bar_positions = haplotype_counts['bin_center']
plt.plot(bar_positions, bar_heights, 'b-', alpha=0.5, linewidth=1.1)

# Add dots at the middle of each bar
plt.scatter(bar_positions, bar_heights, color='blue', s=10, zorder=3)

plt.xlim(0, max_pct)  # Use the same x-axis limits as the density plots
plt.xlabel('Haplotype Sequence Length\n(% of Reference)')
plt.ylabel('Unique\nHaplotypes')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
# Set labels and titles
g.set_xlabels('Haplotype Sequence Length\n(% of Reference)')
g.set_ylabels('VEP Score')
g.set_titles('{row_name}')
g.tight_layout()
plt.show()


Filter out variants with less than 50% of the haplotype sequence length.

In [ ]:
vep_df = vep_df.loc[vep_df['haplotype_sequence_len_pct']>.5]

### Count number of pathogenic variants already in the haplotype

In [ ]:
vep_df = hs.add_haplotype_diffs(vep_df,
                                haplotypes=haplotypes, 
                                force=True)
vep_df.head()

In [ ]:
# Melt the diff columns to create a long-format dataframe
diff_cols = [col for col in vep_df.columns if col.startswith('diff_')]
plot_data = vep_df.loc[(vep_df['VEP'].notna()) & (vep_df['mutant_in_haplotype']==False)].melt(
    id_vars=['model_location', 'protein', 'haplotype', 'clinsig', 'VEP','haplotype_sequence_len'],
    value_vars=diff_cols,
    var_name='diff_type',
    value_name='diff_count'
)
plot_data['diff_count_scaled'] = plot_data['diff_count'] / plot_data['haplotype_sequence_len']

plot_data = plot_data.loc[plot_data['diff_count'] > 0]

# Create a categorical column with the defined order
plot_data['clinsig'] = pd.Categorical(
    plot_data['clinsig'], 
    categories=list(utils.get_clinsig_palette().keys())[::-1], 
    ordered=True
)
# Clean up the diff_type column by removing the 'diff_' prefix
plot_data['diff_type'] = plot_data['diff_type'].str.replace('diff_', '')
plot_data['diff_type'] = pd.Categorical(
    plot_data['diff_type'], 
    categories=['benign','unknown', 'possibly damaging', 'probably damaging'], 
    ordered=True
)
# Sort the dataframe by the ordered categorical column
plot_data = plot_data.sort_values(['clinsig','diff_type'])

# # Group by relevant columns and calculate mean VEP score
plot_data = plot_data.groupby(['model_location', 'protein', 'haplotype','clinsig', 'diff_type','diff_count_scaled'], observed=True).agg({'VEP': 'mean'}).reset_index()

# plot_data = plot_data.loc[plot_data['diff_type'] == 'probably damaging']

# plot_data = plot_data.groupby(['clinsig', 'diff_type'], observed=True).sample(1000, random_state=42).reset_index(drop=True)

print("plot data shape: ", plot_data.shape)

g = sns.FacetGrid(data=plot_data, 
                  col='diff_type',
                   row='clinsig',
                   margin_titles=True,
                   height=4, aspect=1.2)
g.map_dataframe(sns.kdeplot, 
                x='diff_count_scaled', 
                y='VEP',
                fill=True,
                thresh=0, levels=50, cmap="mako",
                bw_adjust=0.5, 
                # clip=[(None,max_pct),(min_vep, None)],
                extend="both",
                alpha=1.0
                )
g.map_dataframe(sns.regplot,
                x="diff_count_scaled",
                y="VEP", 
                scatter=False,
                lowess=True,
                line_kws={"color": "white"}
                )
g.add_legend()

In [ ]:
# # Remove any existing hap_mutations columns if they exist
# force = True
# if any([col for col in vep_df.columns if col.startswith('n_')]) and force:
#     vep_df = vep_df.drop(columns=[col for col in vep_df.columns if col.startswith('n_')])

# # Pivot the table to create columns for each clinsig category
# hap_mutations = vep_df.loc[vep_df['mutant_in_haplotype']==True].pivot_table(
#     index='haplotype',
#     columns='clinsig',
#     values='mutant',
#     aggfunc='nunique',
#     fill_value=0
# ).add_prefix('nWT_')

# # Convert all columns to integer type
# hap_mutations = hap_mutations.astype(int)
# hap_mutations
# #merge back into vep_df
# vep_df = vep_df.merge(hap_mutations, left_on='haplotype', right_index=True, how='left')
# vep_df[hap_mutations.columns] = vep_df[hap_mutations.columns].fillna(0)
# vep_df.head()
# # Plot haplotype_sequence_len_pct vs VEP as point density with regression line

In [ ]:
# haplotypes = hs.get_haplotypes(
#     cache_only=True 
#     ) 
# vep_df['protein_id'] = vep_df['haplotype'].str.split(':').str[0]

# vep_df = hs.add_haplotype_freqs(df= vep_df, 
#                                 haplotypes = haplotypes, 
#                                 haplotype_col = "haplotype")
# vep_df.head() 

In [ ]:
# seq_check_df, clinsig_counts, mutant_in_haplotype = va.report_vep(vep_df)
# display(seq_check_df, clinsig_counts, mutant_in_haplotype)

## Plots: All Proteins

Filter out variants with less than 50% of the haplotype sequence length.

In [ ]:
vep_df = vep_df.loc[vep_df['haplotype_sequence_len_pct']>.5]

### Density plots

In [ ]:
va.plot_vep_density(vep_df, 
                     multiple=['layer','stack','fill'][0],
                     row='scoring_strategy',
                     col='model_location',    
                     legend_y=1.1,
                     aspect=1,
                     save_path="results/plots/vep_density.png"
                     )

### Variance plots

In [ ]:
vep_variance = va.plot_vep_variance(vep_df, 
                                    return_df=True,
                                    showfliers=False,
                                    col='model_location',
                                    row='scoring_strategy',
                                    # save_path="results/plots/vep_variance.png",
                                    groupby_cols = ['model_location','protein',
                                                    'clinsig','mutant','scoring_strategy'],
                                    height=3
                                    )

## Plots: Single Protein

### Violin plots: wt-marginals

In [ ]:
va.plot_vep_violin(vep_df, 
                #    model_location="esm2_t33_650M_UR50D",
                   max_proteins=1,
                   max_models=1,
                   save_path="results/plots/vep_violin;esm2_t33_650M_UR50D;wt-marginals.png",
                   scoring_strategy="wt-marginals")

### Violin plots: masked-marginals

In [ ]:
va.plot_vep_violin(vep_df, 
                   model_location="esm2_t33_650M_UR50D",
                    max_proteins=1,
                    max_models=1,
                    save_path="results/plots/vep_violin;esm2_t33_650M_UR50D;masked-marginals.png",
                    scoring_strategy="masked-marginals")

### Haplotype Representiveness

In [ ]:
vep_df['VEP_percentile'] = vep_df.groupby(['model_location', 'protein', 'mutant','scoring_strategy'])['VEP'].rank(pct=True)
vep_df[['VEP_percentile']]

### Reference Representativeness: Percentiles

In [ ]:
vep_df = hs.add_haplotype_freqs(df=vep_df)
vep_df.head()

In [ ]:
vep_df['Top Superpopulation'] = vep_df['top_superpop'].str.split(':').str[-1]
va.plot_vep_percentiles(vep_df.loc[vep_df['haplotype_sequence_len_pct'] > .5], 
                        col='model_location',
                        x="Top Superpopulation",
                        row='scoring_strategy',
                         groupby_cols = ['model_location','protein','clinsig','mutant','scoring_strategy',
                                         "Top Superpopulation"],
                        height=5,
                        aspect=2,
                        save_path="results/plots/vep_percentiles.png")

In [ ]:
vep_df.loc[vep_df['is_ref']==True]['VEP_percentile'].describe()

Get genes where the reference is the LEAST representitive

In [ ]:
vep_df.loc[:,'VEP_percentile_deivation'] = abs(vep_df['VEP_percentile'] - 50)
most_deviant_ref = vep_df.loc[vep_df['is_ref']==True].groupby(['model_location', 'scoring_strategy','protein']).agg({'VEP_percentile_deivation':['count','min','max','mean','median']}).sort_values(('VEP_percentile_deivation','mean'), ascending=False).reset_index()
most_deviant_ref.head(10)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a FacetGrid to facet by scoring_strategy
g = sns.violinplot(data=vep_df.loc[vep_df['protein'].isin(most_deviant_ref['protein'].unique()[:12])],
                   x='is_ref',
                   y='VEP',
                   hue='scoring_strategy',
                   )
plt.show()

In [ ]:
from gprofiler import GProfiler
gp = GProfiler(return_dataframe=True)
gp_df = gp.profile(organism='hsapiens',
           background=vep_files['protein'].unique().tolist(),
           query=most_deviant_ref['protein'].unique().tolist()[:50]
           )
gp_df


## Precision-Recall 

### Add edit distance metrics

In [ ]:
# Add the number of edits (variants relative to the reference) per haplotype
vep_df = utils.add_edits(vep_df)

# Scale edits by protein sequence length
seq_lens = {seq:len(seq) for seq in vep_df['protein_sequence']}
vep_df['protein_sequence_len'] = vep_df['protein_sequence'].map(seq_lens)
vep_df['edits_scaled'] = vep_df['edits'] / vep_df['protein_sequence_len']
vep_df['edits_clipped'] = vep_df['edits'].clip(upper=20)
vep_df.loc[:,'edits_scaled_rounded'] = vep_df['edits_scaled'].round(3)

# vep_df['edits_scaled_quartiles'] = pd.qcut(vep_df.groupby(['protein','mutant'])['edits_scaled'], q=10, labels=False)
# vep_df['edits_scaled_quartiles'].value_counts()

Filter to only include variants with high-confidence annotations.

In [ ]:
print(vep_df.groupby("clinsig")['CLNREVSTAT'].value_counts())

vep_qc = vep_df.loc[(vep_df['clinsig'].isin(["path","benign"])) &\
                    (vep_df['CLNREVSTAT'].isin(['reviewed_by_expert_panel','practice_guideline']))]

In [ ]:
vep_pr = va.compute_precision_recall(vep_qc,
                                     groupby_cols = ['model_location', 'scoring_strategy',
                                                     'protein',
                                                     'is_ref', 
                                                     'protein_sequence_len', 'haplotype',
                                                    #  'edits',
                                                    'edits_scaled_rounded'
                                                     ],
                                     agg_cols = ['protein','protein_sequence_len', 'haplotype'],
                                     x='VEP',
                                     y='DMS_bin_score')


In [ ]:
print(vep_pr.shape)
vep_pr.head()

In [ ]:
vep_pr_ref_only = va.compute_precision_recall(vep_qc.loc[vep_qc['is_ref']==True],
                                     groupby_cols = ['model_location', 'scoring_strategy','protein'],
                                     agg_cols = ['protein'],
                                     x='VEP',
                                     y='DMS_bin_score')
vep_pr_ref_only['group'] = 'Ref only'
vep_pr_all = va.compute_precision_recall(vep_qc,
                                     groupby_cols = ['model_location', 'scoring_strategy','protein'],
                                     agg_cols = ['protein'],
                                     x='VEP',
                                     y='DMS_bin_score')
vep_pr_all['group'] = 'All'
vep_pr_grouped = pd.concat([vep_pr_ref_only, vep_pr_all])
vep_pr_grouped


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a FacetGrid to facet by scoring_strategy
g = sns.FacetGrid(data=vep_pr_grouped,
                  row="scoring_strategy",
                  margin_titles=True,
                  height=5, 
                  aspect=1)

# Map the barplot to each facet
g.map_dataframe(sns.barplot, 
                x="model_location", 
                y="auc", 
                hue="group",
                palette='Blues')

# Fix x-axis labels - apply rotation to each subplot
for ax in g.axes.flat:
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# Adjust layout and add titles
g.set_axis_labels("Model", "AUC")
# _rm_subplot_prefixes(g)

# Add a legend for the hue variable
g.add_legend(title="Dataset", loc='upper center', ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
va.plot_auc_by_edits(vep_pr, #vep_pr.loc[vep_pr['edits'] < 20],
                      col="model_location",
                      row="scoring_strategy", 
                      style="is_ref",
                     x="edits_scaled_rounded", 
                     save_path="results/plots/vep_pr_auc_by_edits.png")

In [ ]:
va.plot_precision_recall(vep_pr,
                         style='is_ref',
                         
                         save_path="results/plots/precision_recall.png",
                         hue='scoring_strategy')

### Pairwise correlations between haplotypes
Compute pairwise correlations between all haplotypes for a given protein.

In [ ]:
# Group by protein and pivot to get haplotype columns
tqdm.pandas(desc="Pivoting VEP")
vep_pivot = vep_df.groupby('protein').progress_apply(lambda x: pd.pivot_table(x,
                                                                    index=['protein', 'mutant'],
                                                                    columns='edits_clipped',
                                                                    values='VEP',
                                                                    # Only include haplotypes for this protein
                                                                    dropna=True))
vep_pivot


In [ ]:
vep_pivot.index.get_level_values('mutant').value_counts()

# Calculate percentage of non-NA values for each row
pct_nonna = (vep_pivot.notna().sum(axis=0) / vep_pivot.shape[0] * 100).round(2)

pct_nonna.describe()


In [ ]:
vep_pivot.index.get_level_values(0).nunique()

In [ ]:
# Compute correlations within each protein
vep_corr = vep_pivot.groupby('protein').apply(lambda x: x.corr())
import matplotlib.pyplot as plt
import seaborn as sns
# Create list to store correlation matrices
corr_matrices = []

max_proteins = 3
# Get correlation matrix for each protein
for protein in vep_corr.index.get_level_values(0).unique()[:max_proteins]:
    corr_matrix = vep_corr.loc[protein]
    # Remove columns that are all NA
    corr_matrix = corr_matrix.dropna(axis=1, how='all')
    # Remove rows that are all NA 
    corr_matrix = corr_matrix.dropna(axis=0, how='all')
    plt.figure(figsize=(12,10))
    sns.heatmap(corr_matrix,
                cmap='Reds', 
                # center=vep_corr.mean().mean(),
                # vmin=vep_corr.min().min(),
                # vmax=vep_corr.max().max()
                )
    n_mutations = len(vep_pivot.loc[protein].index)
    plt.title(f'VEP Correlations Between Haplotypes\n{protein} ({n_mutations} mutations)')
    plt.tight_layout()
    plt.show()
    corr_matrices.append(corr_matrix)

# Calculate mean correlation matrix
mean_corr = sum(corr_matrices) / len(corr_matrices)

# Plot mean correlation heatmap
plt.figure(figsize=(12,10))
sns.heatmap(mean_corr,
            cmap='Reds',
            )
plt.title(f'Mean VEP Correlations Between Haplotypes\n{vep_pivot.index.get_level_values(0).nunique()} proteins, {len(vep_pivot.columns)} haplotypes, {len(vep_pivot.index)} mutations')
plt.tight_layout()
plt.show()


## Define decision boundaries between benign and pathogenic variants

### Identify cases where the VEP of pathogenic variants overlaps with the VEP of benign variants 

Within the same model_location, metric, protein


In [ ]:
# Group by protein, model_location, and scoring_strategy
# For each group, check if there's overlap between pathogenic and benign VEP distributions
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
overlap_results = []
groupby_cols = ['protein', 'model_location', 'scoring_strategy']

for (protein, model, scoring), group in tqdm(vep_df.groupby(groupby_cols), 
                                             total=len(vep_df.groupby(groupby_cols))):
    # Get benign and pathogenic variants
    benign_variants = group[group['clinsig'].isin(['benign', 'likely_benign'])]
    path_variants = group[group['clinsig'].isin(['path', 'likely_path'])]
    
    # Only proceed if we have both types of variants
    if len(benign_variants) > 0 and len(path_variants) > 0:
        # Get min and max VEP for each category
        benign_min = benign_variants['VEP'].min()
        benign_max = benign_variants['VEP'].max()
        path_min = path_variants['VEP'].min()
        path_max = path_variants['VEP'].max()
        
        # Check for overlap
        has_overlap = (path_min <= benign_max) and (benign_min <= path_max)
        
        # Calculate overlap range if there is overlap
        overlap_range = None
        if has_overlap:
            overlap_min = max(benign_min, path_min)
            overlap_max = min(benign_max, path_max)
            overlap_range = overlap_max - overlap_min
        
        # Store results
        overlap_results.append({
            'protein': protein,
            'model_location': model,
            'scoring_strategy': scoring,
            'benign_count': len(benign_variants),
            'path_count': len(path_variants),
            'benign_min': benign_min,
            'benign_max': benign_max,
            'path_min': path_min,
            'path_max': path_max,
            'has_overlap': has_overlap,
            'overlap_range': overlap_range
        })

# Convert to DataFrame
overlap_df = pd.DataFrame(overlap_results)

# Display summary
print(f"Total protein-model-scoring combinations: {len(overlap_df)}")
print(f"Combinations with VEP overlap between pathogenic and benign variants: {overlap_df['has_overlap'].sum()}")
print(f"Percentage of combinations with overlap: {100 * overlap_df['has_overlap'].mean():.2f}%")

# Display proteins with overlap
if len(overlap_df[overlap_df['has_overlap']]) > 0:
    print("\nProteins with VEP overlap between pathogenic and benign variants:")
    display(overlap_df[overlap_df['has_overlap']].sort_values('overlap_range', ascending=False))

# Visualize one example of overlap if any exists
if overlap_df['has_overlap'].any():
    # Get the protein with the largest overlap
    example_protein = overlap_df.loc[overlap_df['has_overlap'], 'protein'].iloc[0]
    example_model = overlap_df.loc[overlap_df['has_overlap'], 'model_location'].iloc[0]
    example_scoring = overlap_df.loc[overlap_df['has_overlap'], 'scoring_strategy'].iloc[0]
    
    # Filter data for this protein
    example_data = vep_df[(vep_df['protein'] == example_protein) & 
                          (vep_df['model_location'] == example_model) &
                          (vep_df['scoring_strategy'] == example_scoring)]
    
    # Create plot
    plt.figure(figsize=(10, 6))
    sns.histplot(data=example_data, x='VEP', hue='clinsig', 
                 palette=utils.get_clinsig_palette(), 
                 bins=100,
                 element='step', common_norm=False, stat='density')
    plt.title(f"VEP Distribution for {example_protein}\nModel: {example_model}, Scoring: {example_scoring}")
    plt.xlabel("Variant Effect Prediction (VEP)")
    plt.ylabel("Density")
    plt.legend(title="Clinical Significance")
    plt.show()


In [ ]:
import src.vep_gmm as vg

gmm_df = vg.train_vep_gmm(vep_df)

In [ ]:
boundary_crossing_df = vg.get_decision_boundaries(gmm_df, vep_df, plot=False)
boundary_crossing_df.head()

Find variants where the VEP of a pathogenic variant overlaps with the VEP of a benign variant for some haplotypes but not others

### Embed VEP results in 2D space

In [ ]:
variant_data = vg.get_example_data_for_decision_boundary(boundary_crossing_df, vep_df, top_n=3)[1]
variant_data = hs.add_haplotype_freqs(variant_data)

In [ ]:
vep_df.groupby(['scoring_strategy'])['VEP'].describe()

In [ ]:
print(vep_df.shape)
print(vep_df.model_location.nunique(),"model(s)")
print(vep_df.protein.nunique(),"protein(s)")
print(vep_df.haplotype.nunique(),"haplotypes(s)")
print((vep_df.protein + '_' + vep_df.mutant).nunique(),"mutantation(s)")
print(vep_df.scoring_strategy.nunique(),"scoring strategy(ies)")

vep_df.head()

In [ ]:
freq_cols = [col for col in variant_data.columns if col.startswith('freq_')]
# superpops = variant_data['top_superpop'].unique().tolist()
# freq_cols = [col for col in freq_cols if col.endswith(tuple(superpops))]


# Run PCA on the population frequency data
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(variant_data.set_index('haplotype').loc[:,freq_cols].fillna(0))

# Apply PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Create a DataFrame with the PCA results
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'], 
                      index=variant_data['haplotype'].tolist()
                      ).merge(variant_data, 
                              left_index=True, 
                              right_on='haplotype'
                              ).set_index('haplotype')
# Color by VEP
pca_df['Top\nSuperpopulation'] = pca_df['top_superpop'].str.split(':').str[-1]
pca_df['Superpopulation-\nSpecific'] = pca_df['nonzero_superpop_freqs']==1

pca_df.sort_values(['Top\nSuperpopulation','VEP'],inplace=True)
# Apply MinMax scaling to normalize VEP scores between 0 and 1
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
pca_df['VEP\n(normalized)'] = 1- scaler.fit_transform(pca_df[['VEP']]).flatten()
# Plot the PCA results using seaborn
plt.figure(figsize=(8, 5))  # Increased width to accommodate legend outside
# Get the decision boundary value from the data if available
if 'decision_boundary' in pca_df.columns:
    decision_boundary = pca_df['decision_boundary'].iloc[0]
else:
    # Use a default value around -6 based on the context
    decision_boundary = -6

# Calculate the range for the color palette normalization
vep_min = pca_df['VEP'].min()
vep_max = pca_df['VEP'].max()
# Ensure the decision boundary is in the middle
palette_range = (2*decision_boundary - vep_max, vep_max)

sns.scatterplot(data=pca_df, 
                x='PC1',
                y='PC2', 
                size="freq_1000GENOMES:phase_3:ALL",
                # size="VEP\n(normalized)",
                # sizes=(50, 200),
                # hue='Top\nSuperpopulation',
                # palette=utils.get_superpop_palette(),
                hue='VEP',
                palette="bwr_r",
                # palette=sns.diverging_palette(h_neg=10, h_pos=240, s=100, l=60, 
                #                               as_cmap=True, center="dark"),
                hue_norm=palette_range,  # Dynamically set the middle of the palette to the decision boundary
                style='Superpopulation-\nSpecific',
                alpha=0.85,
                edgecolor='grey',
                )
plt.title(f'PCA of Haplotype Population Frequencies\nProtein: {pca_df.iloc[0]["protein"]}; Variant: {pca_df.iloc[0]["mutant"]}; {pca_df.shape[0]} haplotypes')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance explained)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance explained)')

# Move legend outside the plot to the right
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, title=f"VEP (boundary={decision_boundary:.2f})")

# Add annotations with repel labels
from adjustText import adjust_text
# Create a list to store annotation objects
texts = []
label_topn = 10
pca_df.sort_values(["top_superpop_freq",'VEP'], ascending=False, inplace=True)
for i, (haplotype, row) in enumerate(pca_df.iterrows()):  # Annotate first 10 points
    if i > label_topn:
        break
    txt = haplotype.split(':')[-1]
    if 'top_superpop' in row.index:
        txt = f"{txt} ({row['top_superpop'].split(':')[-1]})"
    # Create annotation and add to list
    texts.append(plt.text(x=pca_df['PC1'].iloc[i], 
                          y=pca_df['PC2'].iloc[i], 
                          s=txt,
                          fontsize=8))

# Apply the repel algorithm to prevent overlapping labels
adjust_text(texts, arrowprops=dict(arrowstyle='->', color='black', lw=0.5, alpha=.75), expand=(1.2, 3))

# Add a black ring around the reference haplotype
ref_data = pca_df.loc[pca_df.is_ref==True]
if len(ref_data) > 0:
    plt.scatter(
        ref_data['PC1'], 
        ref_data['PC2'], 
        s=250,  # Slightly larger than the largest point
        facecolors='none', 
        edgecolors='black', 
        linewidths=1,
        linestyle='--',
        alpha=.5,
        zorder=10  # Ensure it's drawn on top of other points
    )

plt.grid(alpha=0.3)
plt.tight_layout()  # Adjust layout to make room for the legend
plt.show()

# Print variance explained
print(f"Variance explained by PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"Variance explained by PC2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_[:2]):.2%}")


In [ ]:
# Get proteins where VEP is 0
print(vep_df.loc[vep_df['VEP']==0]['protein'].nunique())

# Get proteins where VEP is NA
print(vep_df.loc[vep_df['VEP'].isna()]['protein'].nunique())